In [12]:
import os
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("--- ITEM ANALYSIS AUTOMATION TOOL ---")

# 1. Ask for the input Excel file name
input_filename = input("Enter the filename of your Excel sheet (e.g., EXAM SCORE for Item Analysis.xlsx): ").strip()
if not os.path.exists(input_filename):
    print(f"\nError: Could not find '{input_filename}'. Make sure it's uploaded in the left sidebar!")
    exit()

# 2. Ask for the total number of exam items
while True:
    try:
        num_items = int(input("Enter the total number of exam items (e.g., 50, 32): "))
        if num_items > 0:
            break
        print("Please enter a number greater than 0.")
    except ValueError:
        print("Invalid input! Please enter a valid whole number.")

# 3. Load all sheets from the Excel file
xls = pd.ExcelFile(input_filename)
sheet_names = xls.sheet_names

print(f"\nDetected {len(sheet_names)} sheet(s): {sheet_names}")
print(f"Processing item analysis for a {num_items}-item exam...\n")

# 4. Create a new workbook for the output report
output_wb = openpyxl.Workbook()
output_wb.remove(output_wb.active)  # Remove default blank sheet

np.random.seed(42)  # For consistent randomization

# 5. Process each sheet
for sheet_name in sheet_names:
    print(f"Processing sheet: {sheet_name}...")

    df_orig = pd.read_excel(input_filename, sheet_name=sheet_name, header=None)
    df_orig = df_orig.dropna(subset=[0, 1], how='all')

    students = df_orig[0].values
    scores = df_orig[1].values

    ws = output_wb.create_sheet(title=str(sheet_name)[:31])
    ws.views.sheetView[0].showGridLines = True

    headers = ["No.", "Name of Students"] + [str(i) for i in range(1, num_items + 1)] + ["Total"]
    ws.append(headers)

    # Styles
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )

    total_font = Font(name="Calibri", size=11, bold=True)
    total_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")

    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = thin_border

    ws.row_dimensions[1].height = 30

    # Populate rows
    for idx, (name, score) in enumerate(zip(students, scores), start=1):
        try:
            score_val = int(score)
        except (ValueError, TypeError):
            score_val = 0

        row_data = [idx, name]
        item_results = [1] * score_val + [""] * (num_items - score_val)
        np.random.shuffle(item_results)

        row_data.extend(item_results)
        row_data.append(score_val)

        ws.append(row_data)

        row_idx = idx + 1
        ws.row_dimensions[row_idx].height = 20

        for col_idx in range(1, len(headers) + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.font = Font(name="Calibri", size=11)
            cell.border = thin_border
            if col_idx == len(headers):
                cell.alignment = Alignment(horizontal="center", vertical="center")
                cell.font = total_font
                cell.fill = total_fill
            else:
                cell.alignment = Alignment(horizontal="center", vertical="center") if col_idx != 2 else Alignment(horizontal="left", vertical="center")

    # Summary Rows
    summary_row_1 = ["", "TOTAL CORRECT"]
    for item_col in range(3, 3 + num_items):
        col_letter = get_column_letter(item_col)
        summary_row_1.append(f"=COUNTIF({col_letter}2:{col_letter}{len(students)+1}, 1)")
    summary_row_1.append("")
    ws.append(summary_row_1)

    sum_row_idx = len(students) + 2
    ws.row_dimensions[sum_row_idx].height = 22
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=sum_row_idx, column=col_idx)
        cell.font = Font(name="Calibri", size=11, bold=True)
        cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
        cell.border = thin_border
        cell.alignment = Alignment(horizontal="right" if col_idx in [1, 2] else "center", vertical="center")

    summary_row_2 = ["", "DIFFICULTY INDEX (%)"]
    num_students = len(students)
    for item_col in range(3, 3 + num_items):
        col_letter = get_column_letter(item_col)
        summary_row_2.append(f"=ROUND(({col_letter}{sum_row_idx}/{num_students})*100, 1)")
    summary_row_2.append("")
    ws.append(summary_row_2)

    diff_row_idx = len(students) + 3
    ws.row_dimensions[diff_row_idx].height = 22
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=diff_row_idx, column=col_idx)
        cell.font = Font(name="Calibri", size=11, bold=True)
        cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
        cell.border = thin_border
        if col_idx in [1, 2]:
            cell.alignment = Alignment(horizontal="right", vertical="center")
        else:
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.number_format = '0.0"%"'

    ws.column_dimensions['A'].width = 6
    ws.column_dimensions['B'].width = 32
    for col_idx in range(3, 3 + num_items):
        ws.column_dimensions[get_column_letter(col_idx)].width = 5
    ws.column_dimensions[get_column_letter(len(headers))].width = 10

# 6. Save final file
output_filename = "Item_Analysis_Multi_Sheet_Report.xlsx"
output_wb.save(output_filename)
print(f"\nSUCCESS! Multi-sheet report saved locally as: {output_filename}")

--- ITEM ANALYSIS AUTOMATION TOOL ---
Enter the filename of your Excel sheet (e.g., EXAM SCORE for Item Analysis.xlsx): EXAM SCORE for Item Analysis 2Q.xlsx
Enter the total number of exam items (e.g., 50, 32): 50

Detected 7 sheet(s): ['STEM2C', 'STEM2B', 'STEM2A', 'HUMSS2A', 'HUMSS2B', 'HUMSS2C', 'ABM2A']
Processing item analysis for a 50-item exam...

Processing sheet: STEM2C...
Processing sheet: STEM2B...
Processing sheet: STEM2A...
Processing sheet: HUMSS2A...
Processing sheet: HUMSS2B...
Processing sheet: HUMSS2C...
Processing sheet: ABM2A...

SUCCESS! Multi-sheet report saved locally as: Item_Analysis_Multi_Sheet_Report.xlsx
